# Inference demo: protocol detector on freshly generated spectrograms

Loads the 4-protocol YOLO detector (`runs/detector_run2/best.pt`, or `DETECTOR_WEIGHTS`; a one-class detector also works) and runs it on scenes made with the **same generator** as `01_create_dataset.ipynb` (its code cells are executed here, so nothing is copied). Nothing is read from the dataset folder and nothing is trained.

| Section | Question |
|---|---|
| 1 | Does it find and name the protocols in 5 random scenes (with distractors AM / CW / POCSAG in some)? |
| 2 | How do recall and protocol accuracy depend on SNR, per protocol? |
| 3 | DMR next to P25 (the hard pair): what does the detector say, and does the IQ refinement help? |
| 4 | What does it say about signals it never saw (NBFM, WFM, GSM) or must ignore (AM, CW, POCSAG)? |
| 5 | IQ images: how many false alarms does the mirror filter remove when the receiver has a poor image rejection? |

On every spectrogram: solid boxes = ground truth with protocol and SNR, dashed red boxes = detector output with its protocol and confidence.

In [ ]:
import json, os, sys
from pathlib import Path
import numpy as np

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
nb_src = json.loads((ROOT / "01_create_dataset.ipynb").read_text())
# generator cells: imports, config, classes, signals, envelopes, receiver, scene composer, sessions (without their demo plots)
for idx in (1, 3, 5, 7, 9, 11, 13, 15):
    src = "".join(nb_src["cells"][idx]["source"])
    src = src.split("_demo = make_specs")[0].split("# quick look at the envelopes")[0]
    exec(compile(src, f"01_create_dataset.ipynb[cell {idx}]", "exec"), globals())
print("generator ready:", [c[1] for c in CLASS_TABLE], "| distractors:", DISTRACT, "| held out:", UNKNOWN)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle
from IPython.display import display

from train_detector import Detector
from spectrum_common import box_iou_matrix, match_detections

_default = ROOT / "runs/detector_run2/best.pt"
WEIGHTS = Path(os.environ.get("DETECTOR_WEIGHTS", _default if _default.exists() else ROOT / "runs/detector_run1/best.pt"))
det = Detector(WEIGHTS, conf=0.25)                                  # mirror filter on (drops IQ-image copies of stronger signals)
det_raw = Detector(WEIGHTS, conf=0.25, mirror_filter=False)         # the same weights without it, for section 5
MULTI = not det.single_class
print("weights", WEIGHTS, "| imgsz", det.imgsz, "| device", det.device, "| 4-protocol model" if MULTI else "| one-class model")

SEED = None                                                         # set an int for reproducible scenes
rng = np.random.default_rng(SEED)


def new_scene(seed, force=None, irr_db=None):
    """One scene. force = [dict(cls, snr_db, fc_hz)] places exactly those signals; otherwise a random 'general' scene."""
    spec = make_specs("val", 0, 1)[0]
    spec.update(seed=int(seed), scene_id=f"demo_{int(seed) % 10**6}", mode="general")
    spec["session"] = session_for_group(SET_IDS["val"], int(seed) % 30, "train")
    if irr_db is not None:
        spec["session"]["irr_db"] = float(irr_db)
    if force:
        spec["force"] = force
    sc = make_scene(spec)
    sc["db"] = dequantize_db(sc["spec"])
    return sc


def box_to_axes(x0, y0, x1, y1):
    f0 = (x0 - CFG.n_fft / 2 - 0.5) * DF / 1e3
    f1 = (x1 - CFG.n_fft / 2 + 0.5) * DF / 1e3
    t0 = (y0 * CFG.hop + CFG.n_fft / 2 - CFG.hop / 2) / FS * 1e3
    t1 = (y1 * CFG.hop + CFG.n_fft / 2 + CFG.hop / 2) / FS * 1e3
    return f0, f1, t0, t1


def draw(ax, sc, preds, title, xlim=None, fs=9):
    plot_scene(sc, ax=ax)
    [t.remove() for t in list(ax.texts)]
    for r in sc["signals"]:
        f0, f1, t0, t1 = box_to_axes(r["box_x0"], r["box_y0"], r["box_x1"], r["box_y1"])
        ax.text(f0, t0, f'{r["class_name"]}, {r["snr_db"]:.0f} dB', color=FAM_COLORS[r["family_id"]], fontsize=fs, fontweight="bold", va="bottom",
                bbox=dict(fc="black", alpha=0.5, pad=1, lw=0))
    for d in preds:
        f0, f1, t0, t1 = box_to_axes(*d["box"])
        ax.add_patch(Rectangle((f0, t0), f1 - f0, t1 - t0, fill=False, ec="red", ls="--", lw=2))
        name = FAMILIES[d["cls"]] if MULTI else "signal"
        ax.text(f1, t1, f"{name} {d['score']:.2f}", color="white", fontsize=fs, va="top", bbox=dict(fc="red", alpha=0.6, pad=1, lw=0))
    if xlim is not None:
        ax.set_xlim(*xlim)
    ax.set_title(title, fontsize=10)


def known_boxes(sc):
    return [r for r in sc["signals"] if r["family_id"] <= len(FAMILIES)]

## 1. Five random scenes
Random "general" scenes as in the dataset: 1 to 4 labelled signals (random protocols, SNR 5 to 30 dB) and, in 30% of them, an unlabelled distractor (AM, CW or POCSAG) that the detector should ignore. Distractors are drawn in red.

In [ ]:
scenes = []
while len(scenes) < 5:
    sc = new_scene(rng.integers(0, 2**62))
    if len(known_boxes(sc)) > 0:
        scenes.append(sc)
        print(f"scene {len(scenes) - 1}: " + ", ".join(f'{r["class_name"]} @ {r["snr_db"]:.0f} dB' for r in {r["sig_idx"]: r for r in sc["signals"]}.values()))
preds = det.detect_batch([sc["db"] for sc in scenes])
fig, axes = plt.subplots(len(scenes), 1, figsize=(16, 5 * len(scenes)))
for k, (ax, sc, pr) in enumerate(zip(axes, scenes, preds)):
    draw(ax, sc, pr, f"Scene {k}: {len(pr)} detections")
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for k, (sc, pr) in enumerate(zip(scenes, preds)):
    kb = known_boxes(sc)
    gt = [tuple(r[c] for c in ("box_x0", "box_y0", "box_x1", "box_y1")) for r in kb]
    m = match_detections([d["box"] for d in pr], [d["score"] for d in pr], gt, 0.5) if pr and gt else np.full(len(pr), -1)
    found = {int(g): p for p, g in enumerate(m) if g >= 0}
    for j, r in enumerate(kb):
        pred = (FAMILIES[pr[found[j]]["cls"]] if MULTI else "signal") if j in found else "-"
        rows.append(dict(scene=k, protocol=r["class_name"], snr_db=round(r["snr_db"], 1), burst=r["burst_type"], detected=j in found, predicted=pred,
                         correct=(pred == r["class_name"]) if j in found and MULTI else None))
    n_fp = sum(1 for g in m if g < 0)
    if n_fp:
        rows.append(dict(scene=k, protocol="(false alarms)", snr_db=np.nan, burst="", detected=False, predicted=f"{n_fp} extra detection(s)", correct=None))
df = pd.DataFrame(rows)
display(df)
real = df[df.protocol != "(false alarms)"]
print(f"recall {real.detected.mean():.0%}" + (f" | protocol correct for {int(real.correct.fillna(False).sum())} of {int(real.detected.sum())} detected" if MULTI else ""))

## 2. Recall and protocol accuracy against SNR
Each protocol alone in a scene, 10 scenes for every SNR. The SNR is measured over the signal's own occupied bandwidth, so at the same SNR the 7 kHz wide DMR and P25 channels are much fainter per bin than the 1 MHz wide LTE.

In [ ]:
SNRS, N_PER = [5, 8, 12, 16, 20, 25], 10
rows = []
for proto in FAMILIES:
    scs = [new_scene(rng.integers(0, 2**62), force=[dict(cls=proto, snr_db=float(s), fc_hz=float(rng.uniform(-600e3, 600e3)))]) for s in SNRS for _ in range(N_PER)]
    prs = det.detect_batch([s["db"] for s in scs])
    for k, (sc, pr) in enumerate(zip(scs, prs)):
        gt = [tuple(r[c] for c in ("box_x0", "box_y0", "box_x1", "box_y1")) for r in sc["signals"]]
        m = match_detections([d["box"] for d in pr], [d["score"] for d in pr], gt, 0.5) if pr else np.array([], int)
        hit = [(p, g) for p, g in enumerate(m) if g >= 0]
        rows.append(dict(protocol=proto, snr=SNRS[k // N_PER], boxes=len(gt), found=len({g for _, g in hit}),
                         right=sum(1 for p, g in hit if (not MULTI) or FAMILIES[pr[p]["cls"]] == proto), false_alarms=len(pr) - len(hit)))
d = pd.DataFrame(rows).groupby(["protocol", "snr"]).sum()
rec = (d.found / d.boxes).unstack().reindex(FAMILIES)
acc = (d.right / d.boxes).unstack().reindex(FAMILIES)
fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
for a, tab, ttl in ((ax[0], rec, "recall (box found, IoU 0.5)"), (ax[1], acc, "found and the right protocol")):
    im = a.imshow(tab.values, vmin=0, vmax=1, cmap="viridis", aspect="auto")
    a.set_xticks(range(len(SNRS)), SNRS); a.set_yticks(range(len(FAMILIES)), FAMILIES); a.set_xlabel("SNR (dB)"); a.set_title(ttl)
    for i in range(tab.shape[0]):
        for j in range(tab.shape[1]):
            a.text(j, i, f"{tab.values[i, j]:.2f}", ha="center", va="center", color="white" if tab.values[i, j] < 0.6 else "black", fontsize=9)
plt.tight_layout()
plt.show()
print("false alarms per scene:", round(d.false_alarms.sum() / (len(FAMILIES) * len(SNRS) * N_PER), 2))

## 3. DMR next to P25
A continuous DMR downlink and a P25 channel are both 4800-baud 4-level FSK in 12.5 kHz. The spectrogram only differs by about 8% in deviation (648 vs 600 Hz) and by the TDMA burst timing of a DMR mobile. When the dataset and the baselines were built with IQ (`SPECTRUM_STORE_IQ=1`), the GBM can measure the deviation and look for the sync words. This section shows the detector's call for 20 scenes with one of each, and what the **IQ refinement** (GBM restricted to DMR vs P25, from `runs/baselines/feature_clf.joblib`) makes of the same boxes.

In [ ]:
from baselines import FeatureClassifier
gb_file = Path(os.environ.get("BASELINES_DIR", ROOT / "runs/baselines")) / "feature_clf.joblib"
GB = FeatureClassifier.load(gb_file) if gb_file.exists() else None
REFINE = GB is not None and any(n.startswith("nb_") for n in GB.names)
print("IQ refinement:", "available" if REFINE else "not available (train baselines.py on a dataset built with SPECTRUM_STORE_IQ=1)")

iDMR, iP25 = FAMILIES.index("DMR"), FAMILIES.index("P25")
rows, pair_scs, pair_prs = [], [], []
for k in range(20):
    f1, f2 = rng.uniform(-500e3, -60e3), rng.uniform(60e3, 500e3)
    order = [("DMR", f1), ("P25", f2)] if k % 2 == 0 else [("DMR", f2), ("P25", f1)]
    sc = new_scene(rng.integers(0, 2**62), force=[dict(cls=c, snr_db=float(rng.uniform(15, 28)), fc_hz=float(f)) for c, f in order])
    pr = det.detect_batch([sc["db"]])[0]
    pair_scs.append(sc); pair_prs.append(pr)
    gt = [tuple(r[c] for c in ("box_x0", "box_y0", "box_x1", "box_y1")) for r in sc["signals"]]
    m = match_detections([d["box"] for d in pr], [d["score"] for d in pr], gt, 0.5) if pr else np.array([], int)
    for p, g in enumerate(m):
        if g < 0:
            continue
        true, call = sc["signals"][g]["class_name"], FAMILIES[pr[p]["cls"]] if MULTI else "signal"
        refined = None
        if REFINE and MULTI and pr[p]["cls"] in (iDMR, iP25):
            pg = GB.predict_proba(sc["db"], [pr[p]["box"]], iq_to_complex(sc["iq"], sc["scene"]["iq_scale"]))[0]
            refined = "DMR" if pg[iDMR] >= pg[iP25] else "P25"
        rows.append(dict(scene=k, true=true, detector=call, refined=refined, snr_db=round(sc["signals"][g]["snr_db"], 1)))
d = pd.DataFrame(rows)
print(f"detector: DMR right {(d[d.true == 'DMR'].detector == 'DMR').mean():.0%}, P25 right {(d[d.true == 'P25'].detector == 'P25').mean():.0%}")
if REFINE and d.refined.notna().any():
    r = d[d.refined.notna()]
    print(f"after IQ refinement: DMR right {(r[r.true == 'DMR'].refined == 'DMR').mean():.0%}, P25 right {(r[r.true == 'P25'].refined == 'P25').mean():.0%}")
display(pd.crosstab(d.true, d.detector, rownames=["true"], colnames=["detector says"]))
fig, axes = plt.subplots(2, 1, figsize=(16, 9))
for ax, sc, pr in zip(axes, pair_scs[:2], pair_prs[:2]):
    draw(ax, sc, pr, "DMR + P25 in one scene")
plt.tight_layout()
plt.show()

## 4. Signals it never saw, and signals it must ignore
Every signal alone at 20 dB, 10 scenes each. For a *known* protocol the detector should fire with the right name. **AM, CW and POCSAG** are distractors: they were in the training scenes but never labelled, so the detector should stay silent. **NBFM, WFM and GSM** were never seen at all (held out); a detector that fires on them calls them by one of the four protocols, because it has no other word.

In [ ]:
rows = []
for cls in FAMILIES + DISTRACT + UNKNOWN:
    scs = [new_scene(rng.integers(0, 2**62), force=[dict(cls=cls, snr_db=20.0, fc_hz=float(rng.uniform(-600e3, 600e3)))]) for _ in range(10)]
    prs = det.detect_batch([s["db"] for s in scs])
    labels = [FAMILIES[d["cls"]] if MULTI else "signal" for pr in prs for d in pr]
    gt_fire = []
    for sc, pr in zip(scs, prs):
        gt = [tuple(r[c] for c in ("box_x0", "box_y0", "box_x1", "box_y1")) for r in sc["signals"]]
        gt_fire.append(bool(pr) and bool((box_iou_matrix(gt, [d["box"] for d in pr]).max() >= 0.5)))
    role = "known" if cls in FAMILIES else ("distractor (never labelled)" if cls in DISTRACT else "held out (never seen)")
    rows.append(dict(signal=cls, role=role, fires_on_it=f"{np.mean(gt_fire):.0%}", calls_it=pd.Series(labels).value_counts().to_dict() if labels else "-",
                     detections_per_scene=round(len(labels) / len(scs), 2)))
display(pd.DataFrame(rows).set_index("signal"))

## 5. IQ images and the mirror filter
A receiver with an IQ imbalance mirrors every signal about the centre frequency, at the "image rejection" below the signal. With 18 to 22 dB, a strong narrowband signal leaves a copy that a detector can mistake for another transmission. `Detector` removes detections that are the mirror image of a stronger one (`spectrum_common.mirror_suppress`). 30 scenes with a strong DMR or P25 or TETRA signal and an image rejection of 18 to 22 dB, with and without the filter:

In [ ]:
def count(detector, scs):
    tp = fp = fn = 0
    prs = detector.detect_batch([s["db"] for s in scs])
    for sc, pr in zip(scs, prs):
        gt = [tuple(r[c] for c in ("box_x0", "box_y0", "box_x1", "box_y1")) for r in known_boxes(sc)]
        m = match_detections([d["box"] for d in pr], [d["score"] for d in pr], gt, 0.5) if pr and gt else np.full(len(pr), -1)
        tp += int((m >= 0).sum()); fp += int((m < 0).sum()); fn += len(gt) - int((m >= 0).sum())
    return dict(precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1), false_alarms_per_scene=fp / len(scs)), prs


img_scs = [new_scene(rng.integers(0, 2**62), irr_db=float(rng.uniform(18, 22)),
                     force=[dict(cls=str(rng.choice(["DMR", "P25", "TETRA"])), snr_db=float(rng.uniform(24, 30)), fc_hz=float(rng.choice([-1, 1]) * rng.uniform(150e3, 800e3)))])
           for _ in range(30)]
res_raw, prs_raw = count(det_raw, img_scs)
res_flt, prs_flt = count(det, img_scs)
display(pd.DataFrame({"no mirror filter": res_raw, "mirror filter": res_flt}).T.round(3))
k = int(np.argmax([len(a) - len(b) for a, b in zip(prs_raw, prs_flt)]))
fig, axes = plt.subplots(2, 1, figsize=(16, 9))
draw(axes[0], img_scs[k], prs_raw[k], f"without the filter: {len(prs_raw[k])} detections (image rejection {img_scs[k]['scene']['irr_db']:.0f} dB)")
draw(axes[1], img_scs[k], prs_flt[k], f"with the filter: {len(prs_flt[k])} detection(s)")
plt.tight_layout()
plt.show()